# 03 - Evaluacion e Interpretabilidad

Cargamos el modelo entrenado y analizamos:
- Precision real vs prediccion
- Por que ha tomado cada decision (Variable importance + Attention)

In [ ]:
import torch
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (14, 5)
from pathlib import Path
from pytorch_forecasting import TemporalFusionTransformer, TimeSeriesDataSet
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import QuantileLoss
import warnings; warnings.filterwarnings('ignore')

DATA_DIR = Path('../data')
MODELS_DIR = Path('../models')

best_model_path = sorted(MODELS_DIR.glob('tft-gaspredict-*.ckpt'))[-1]
print(f'Cargando: {best_model_path}')
best_tft = TemporalFusionTransformer.load_from_checkpoint(best_model_path)
best_tft.eval()
print('Modelo cargado!')

In [ ]:
# Recrear dataloaders (igual que en el notebook 02)
df = pd.read_parquet(DATA_DIR / 'train_route.parquet')
df['time_idx'] = (pd.to_datetime(df['date']) - pd.to_datetime(df['date']).min()).dt.days
df['id_station'] = df['id_station'].astype(str)
MAX_ENCODER, MAX_PREDICTION = 60, 7
training_cutoff = df['time_idx'].max() - 30

training = TimeSeriesDataSet(
    df[df['time_idx'] <= training_cutoff],
    time_idx='time_idx', target='price_gasolina_95_e5', group_ids=['id_station'],
    max_encoder_length=MAX_ENCODER, max_prediction_length=MAX_PREDICTION,
    static_categoricals=['id_station', 'name', 'municipality'],
    time_varying_known_reals=['time_idx', 'day_of_week', 'day_of_month', 'month', 'is_weekend'],
    time_varying_unknown_reals=['price_gasolina_95_e5', 'price_lag_1d', 'price_lag_7d',
        'brent_usd', 'brent_eur', 'eur_usd', 'brent_lag_7d', 'brent_lag_14d', 'brent_lag_21d', 'brent_lag_28d'],
    target_normalizer=GroupNormalizer(groups=['id_station'], transformation='softplus'),
    add_relative_time_idx=True, add_target_scales=True, add_encoder_length=True,
)
validation = TimeSeriesDataSet.from_dataset(training, df, predict=True, stop_randomization=True)
val_dl = validation.to_dataloader(train=False, batch_size=64, num_workers=0)
print('Dataloaders listos')

In [ ]:
# Variable Importance: que variables usa mas el modelo?
raw = best_tft.predict(val_dl, mode='raw', return_x=True)
interpretation = best_tft.interpret_output(raw.output, reduction='sum')
best_tft.plot_interpretation(interpretation)

In [ ]:
# Atencion temporal: en que dias del pasado se fija mas el modelo?
attention = interpretation['attention']
plt.figure(figsize=(12,4))
plt.plot(attention.mean(0).cpu().numpy())
plt.title('Atencion temporal media: cuanto importa cada dia del pasado?')
plt.xlabel('Dias atras')
plt.ylabel('Peso de atencion')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()